#NBA team winning prediction with Logistic Regression and Season Backtest 
#01 Imports

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss


from pathlib import Path

#02 Load Data

In [2]:
PROJECT_ROOT = Path(".")

reg_matchups = pd.read_csv(
    os.path.join(PROJECT_ROOT, "TeamStatisticsFrom2010RegMatchups.csv")
)

rp_matchups = pd.read_csv(
    os.path.join(PROJECT_ROOT, "TeamStatisticsFrom2010RPMatchups.csv")
)

#03 Create Playoff Indicator

In [3]:
reg_game_ids = set(reg_matchups["gameId"])

rp_matchups["is_playoff"] = (
    ~rp_matchups["gameId"].isin(reg_game_ids)
).astype(int)

matchup_clean = rp_matchups.copy()

matchup_clean["gameDate"] = pd.to_datetime(matchup_clean["gameDate"])

matchup_clean["is_playoff"].value_counts()

0    19130
1     1371
Name: is_playoff, dtype: int64

#04 Split Data

In [4]:
matchup_reg = matchup_clean[matchup_clean["is_playoff"] == 0].copy()
matchup_po  = matchup_clean[matchup_clean["is_playoff"] == 1].copy()

#05 select Features

In [5]:
baseline_features = [
    "win_roll10_diff"
]

full_features = [
    "teamScore_roll10_diff",
    "fieldGoalsPercentage_roll10_diff",
    "threePointersPercentage_roll10_diff",
    "freeThrowsPercentage_roll10_diff",
    "reboundsOffensive_roll10_diff",
    "reboundsDefensive_roll10_diff",
    "assists_roll10_diff",
    "turnovers_roll10_diff",
    "steals_roll10_diff",
    "blocks_roll10_diff",
    "plusMinusPoints_roll10_diff",
    "win_roll10_diff",
    "rest_days_diff",
    "b2b_diff"
]

#06 Model Function with 80/20 time split

In [6]:
def run_logistic_model(data, features, model_name):

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    X = data[features]
    y = data["win"]

    data = data.copy()
    data["gameDate"] = pd.to_datetime(data["gameDate"])

    split_date = data["gameDate"].quantile(0.8)

    train_mask = data["gameDate"] < split_date
    test_mask = data["gameDate"] >= split_date

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
    print("ROC AUC  :", round(roc_auc_score(y_test, y_prob), 4))
    print("Log Loss :", round(log_loss(y_test, y_prob), 4))

    coef = pd.DataFrame({
        "Feature": features,
        "Coef": model.coef_[0]
    }).sort_values("Coef", ascending=False)

    print("\nTop Features:")
    print(coef)

    return model, coef

#07 Regular Season Models

In [7]:
run_logistic_model(matchup_reg, baseline_features, "Regular Season Baseline")


Regular Season Baseline
Accuracy : 0.6302
ROC AUC  : 0.6801
Log Loss : 0.6383

Top Features:
           Feature      Coef
0  win_roll10_diff  0.590413


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
            Feature      Coef
 0  win_roll10_diff  0.590413)

In [8]:
run_logistic_model(matchup_reg, full_features, "Regular Season Full")


Regular Season Full
Accuracy : 0.6495
ROC AUC  : 0.7056
Log Loss : 0.6226

Top Features:
                                Feature      Coef
10          plusMinusPoints_roll10_diff  0.499840
11                      win_roll10_diff  0.147224
9                    blocks_roll10_diff  0.032381
3      freeThrowsPercentage_roll10_diff  0.012520
0                 teamScore_roll10_diff  0.011621
1      fieldGoalsPercentage_roll10_diff  0.007562
6                   assists_roll10_diff  0.006921
8                    steals_roll10_diff  0.001998
12                       rest_days_diff  0.000597
5         reboundsDefensive_roll10_diff -0.011636
7                 turnovers_roll10_diff -0.025381
2   threePointersPercentage_roll10_diff -0.029134
4         reboundsOffensive_roll10_diff -0.048207
13                             b2b_diff -0.117258


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
                                 Feature      Coef
 10          plusMinusPoints_roll10_diff  0.499840
 11                      win_roll10_diff  0.147224
 9                    blocks_roll10_diff  0.032381
 3      freeThrowsPercentage_roll10_diff  0.012520
 0                 teamScore_roll10_diff  0.011621
 1      fieldGoalsPercentage_roll10_diff  0.007562
 6                   assists_roll10_diff  0.006921
 8                    steals_roll10_diff  0.001998
 12                       rest_days_diff  0.000597
 5         reboundsDefensive_roll10_diff -0.011636
 7                 turnovers_roll10_diff -0.025381
 2   threePointersPercentage_roll10_diff -0.029134
 4         reboundsOffensive_roll10_diff -0.048207
 13                             b2b_diff -0.117258)

#08 Playoff Models

In [9]:
run_logistic_model(matchup_po, baseline_features, "Playoff Baseline")


Playoff Baseline
Accuracy : 0.5782
ROC AUC  : 0.6179
Log Loss : 0.6699

Top Features:
           Feature      Coef
0  win_roll10_diff  0.242339


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
            Feature      Coef
 0  win_roll10_diff  0.242339)

In [10]:
run_logistic_model(matchup_po, full_features, "Playoff Full")


Playoff Full
Accuracy : 0.64
ROC AUC  : 0.6477
Log Loss : 0.6507

Top Features:
                                Feature      Coef
10          plusMinusPoints_roll10_diff  0.292331
1      fieldGoalsPercentage_roll10_diff  0.132918
12                       rest_days_diff  0.040530
6                   assists_roll10_diff  0.029484
9                    blocks_roll10_diff  0.020392
7                 turnovers_roll10_diff  0.012546
0                 teamScore_roll10_diff  0.000000
3      freeThrowsPercentage_roll10_diff  0.000000
4         reboundsOffensive_roll10_diff  0.000000
5         reboundsDefensive_roll10_diff  0.000000
8                    steals_roll10_diff  0.000000
11                      win_roll10_diff  0.000000
13                             b2b_diff  0.000000
2   threePointersPercentage_roll10_diff -0.079078


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
                                 Feature      Coef
 10          plusMinusPoints_roll10_diff  0.292331
 1      fieldGoalsPercentage_roll10_diff  0.132918
 12                       rest_days_diff  0.040530
 6                   assists_roll10_diff  0.029484
 9                    blocks_roll10_diff  0.020392
 7                 turnovers_roll10_diff  0.012546
 0                 teamScore_roll10_diff  0.000000
 3      freeThrowsPercentage_roll10_diff  0.000000
 4         reboundsOffensive_roll10_diff  0.000000
 5         reboundsDefensive_roll10_diff  0.000000
 8                    steals_roll10_diff  0.000000
 11                      win_roll10_diff  0.000000
 13                             b2b_diff  0.000000
 2   threePointersPercentage_roll10_diff -0.079078)

#09 Combined Model

In [11]:
run_logistic_model(matchup_clean, baseline_features, "Combined Model")


Combined Model
Accuracy : 0.6297
ROC AUC  : 0.6751
Log Loss : 0.6411

Top Features:
           Feature      Coef
0  win_roll10_diff  0.573834


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
            Feature      Coef
 0  win_roll10_diff  0.573834)

In [12]:
run_logistic_model(matchup_clean, full_features, "Combined Model")


Combined Model
Accuracy : 0.6477
ROC AUC  : 0.6991
Log Loss : 0.6266

Top Features:
                                Feature      Coef
10          plusMinusPoints_roll10_diff  0.502376
11                      win_roll10_diff  0.129078
9                    blocks_roll10_diff  0.036674
1      fieldGoalsPercentage_roll10_diff  0.019289
6                   assists_roll10_diff  0.015896
3      freeThrowsPercentage_roll10_diff  0.012714
12                       rest_days_diff  0.004639
0                 teamScore_roll10_diff  0.004000
8                    steals_roll10_diff  0.000000
5         reboundsDefensive_roll10_diff -0.013211
7                 turnovers_roll10_diff -0.021565
2   threePointersPercentage_roll10_diff -0.037046
4         reboundsOffensive_roll10_diff -0.045100
13                             b2b_diff -0.110712


(LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear'),
                                 Feature      Coef
 10          plusMinusPoints_roll10_diff  0.502376
 11                      win_roll10_diff  0.129078
 9                    blocks_roll10_diff  0.036674
 1      fieldGoalsPercentage_roll10_diff  0.019289
 6                   assists_roll10_diff  0.015896
 3      freeThrowsPercentage_roll10_diff  0.012714
 12                       rest_days_diff  0.004639
 0                 teamScore_roll10_diff  0.004000
 8                    steals_roll10_diff  0.000000
 5         reboundsDefensive_roll10_diff -0.013211
 7                 turnovers_roll10_diff -0.021565
 2   threePointersPercentage_roll10_diff -0.037046
 4         reboundsOffensive_roll10_diff -0.045100
 13                             b2b_diff -0.110712)

#10 Model Function with Season-by-Season Backtest

In [13]:
def run_season_backtest(data, features, name):

    print("\n" + "="*70)
    print(name)
    print("="*70)

    seasons = sorted(data["season"].unique())[1:]

    results = []

    for s in seasons:

        train = data[data["season"] == s - 1]
        test = data[data["season"] == s]

        X_train = train[features]
        X_test = test[features]

        y_train = train["win"]
        y_test = test["win"]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.1,
            random_state=42
        )

        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)
        ll = log_loss(y_test, prob)

        print(f"Season {s}: Acc={acc:.3f}, AUC={auc:.3f}, LogLoss={ll:.3f}")

        results.append([s, acc, auc, ll])

    return pd.DataFrame(results, columns=["season", "acc", "auc", "logloss"])

#11 Regular Season Models Season-by-Season Backtest

In [14]:
run_season_backtest(matchup_reg, baseline_features, "Regular Season Baseline Backtest")



Regular Season Baseline Backtest
Season 2011: Acc=0.638, AUC=0.644, LogLoss=0.644
Season 2012: Acc=0.643, AUC=0.669, LogLoss=0.630
Season 2013: Acc=0.631, AUC=0.675, LogLoss=0.635
Season 2014: Acc=0.657, AUC=0.685, LogLoss=0.627
Season 2015: Acc=0.646, AUC=0.685, LogLoss=0.626
Season 2016: Acc=0.619, AUC=0.640, LogLoss=0.652
Season 2017: Acc=0.624, AUC=0.671, LogLoss=0.636
Season 2018: Acc=0.630, AUC=0.649, LogLoss=0.643
Season 2019: Acc=0.614, AUC=0.658, LogLoss=0.650
Season 2020: Acc=0.596, AUC=0.635, LogLoss=0.661
Season 2021: Acc=0.608, AUC=0.638, LogLoss=0.660
Season 2022: Acc=0.568, AUC=0.595, LogLoss=0.674
Season 2023: Acc=0.598, AUC=0.674, LogLoss=0.655
Season 2024: Acc=0.633, AUC=0.676, LogLoss=0.638
Season 2025: Acc=0.646, AUC=0.700, LogLoss=0.625


,season,acc,auc,logloss
0,2011,0.638384,0.644075,0.643570
1,2012,0.642799,0.669407,0.629939
2,2013,0.630894,0.675270,0.634721
3,2014,0.656911,0.685193,0.626972
4,2015,0.645528,0.685199,0.626179
5,2016,0.618699,0.639633,0.652135
6,2017,0.623577,0.670932,0.636125
7,2018,0.630081,0.649039,0.642909
8,2019,0.613553,0.657614,0.649710
9,2020,0.596296,0.634667,0.660745


In [15]:
run_season_backtest(matchup_reg, full_features, "Regular Season Backtest")


Regular Season Backtest
Season 2011: Acc=0.642, AUC=0.657, LogLoss=0.639
Season 2012: Acc=0.647, AUC=0.676, LogLoss=0.625
Season 2013: Acc=0.631, AUC=0.674, LogLoss=0.635
Season 2014: Acc=0.666, AUC=0.696, LogLoss=0.623
Season 2015: Acc=0.648, AUC=0.702, LogLoss=0.614
Season 2016: Acc=0.626, AUC=0.654, LogLoss=0.646
Season 2017: Acc=0.641, AUC=0.676, LogLoss=0.633
Season 2018: Acc=0.651, AUC=0.672, LogLoss=0.632
Season 2019: Acc=0.619, AUC=0.661, LogLoss=0.648
Season 2020: Acc=0.607, AUC=0.650, LogLoss=0.654
Season 2021: Acc=0.609, AUC=0.641, LogLoss=0.658
Season 2022: Acc=0.600, AUC=0.618, LogLoss=0.665
Season 2023: Acc=0.628, AUC=0.685, LogLoss=0.642
Season 2024: Acc=0.655, AUC=0.710, LogLoss=0.620
Season 2025: Acc=0.656, AUC=0.711, LogLoss=0.616


,season,acc,auc,logloss
0,2011,0.642424,0.656955,0.639338
1,2012,0.646867,0.676168,0.624871
2,2013,0.630894,0.673802,0.634918
3,2014,0.665854,0.696147,0.623422
4,2015,0.647967,0.701502,0.614467
5,2016,0.626016,0.654289,0.645740
6,2017,0.640650,0.676305,0.633282
7,2018,0.651220,0.672195,0.632369
8,2019,0.619048,0.661071,0.647786
9,2020,0.607407,0.650186,0.653537


#12 Playoff Models Season-by-Season Backtest

In [16]:
run_season_backtest(matchup_po, baseline_features, "Playoff Baseline Backtest")


Playoff Baseline Backtest
Season 2011: Acc=0.679, AUC=0.500, LogLoss=0.666
Season 2012: Acc=0.635, AUC=0.500, LogLoss=0.668
Season 2013: Acc=0.562, AUC=0.500, LogLoss=0.689
Season 2014: Acc=0.407, AUC=0.500, LogLoss=0.693
Season 2015: Acc=0.326, AUC=0.500, LogLoss=0.693
Season 2016: Acc=0.570, AUC=0.500, LogLoss=0.684
Season 2017: Acc=0.573, AUC=0.607, LogLoss=0.690
Season 2018: Acc=0.561, AUC=0.500, LogLoss=0.687
Season 2019: Acc=0.506, AUC=0.500, LogLoss=0.693
Season 2020: Acc=0.418, AUC=0.500, LogLoss=0.693
Season 2021: Acc=0.398, AUC=0.500, LogLoss=0.693
Season 2022: Acc=0.411, AUC=0.500, LogLoss=0.693
Season 2023: Acc=0.398, AUC=0.500, LogLoss=0.693
Season 2024: Acc=0.433, AUC=0.500, LogLoss=0.693
Season 2025: Acc=0.440, AUC=0.500, LogLoss=0.693


,season,acc,auc,logloss
0,2011,0.678571,0.500000,0.665954
1,2012,0.635294,0.500000,0.667918
2,2013,0.561798,0.500000,0.689406
3,2014,0.407407,0.500000,0.693147
4,2015,0.325581,0.500000,0.693147
5,2016,0.569620,0.500000,0.683689
6,2017,0.573171,0.607399,0.690113
7,2018,0.560976,0.500000,0.686912
8,2019,0.506329,0.500000,0.693147
9,2020,0.417582,0.500000,0.693147


In [17]:
run_season_backtest(matchup_po, full_features, "Playoff Backtest")


Playoff Backtest
Season 2011: Acc=0.679, AUC=0.500, LogLoss=0.666
Season 2012: Acc=0.635, AUC=0.595, LogLoss=0.662
Season 2013: Acc=0.562, AUC=0.500, LogLoss=0.689
Season 2014: Acc=0.407, AUC=0.500, LogLoss=0.693
Season 2015: Acc=0.535, AUC=0.520, LogLoss=0.703
Season 2016: Acc=0.570, AUC=0.718, LogLoss=0.669
Season 2017: Acc=0.512, AUC=0.567, LogLoss=0.694
Season 2018: Acc=0.561, AUC=0.500, LogLoss=0.687
Season 2019: Acc=0.570, AUC=0.602, LogLoss=0.691
Season 2020: Acc=0.527, AUC=0.536, LogLoss=0.692
Season 2021: Acc=0.548, AUC=0.601, LogLoss=0.691
Season 2022: Acc=0.411, AUC=0.500, LogLoss=0.693
Season 2023: Acc=0.398, AUC=0.500, LogLoss=0.693
Season 2024: Acc=0.400, AUC=0.454, LogLoss=0.696
Season 2025: Acc=0.549, AUC=0.589, LogLoss=0.688


,season,acc,auc,logloss
0,2011,0.678571,0.500000,0.665954
1,2012,0.635294,0.595281,0.662281
2,2013,0.561798,0.500000,0.689406
3,2014,0.407407,0.500000,0.693147
4,2015,0.534884,0.520320,0.703437
5,2016,0.569620,0.718301,0.669295
6,2017,0.512195,0.566810,0.693848
7,2018,0.560976,0.500000,0.686912
8,2019,0.569620,0.602244,0.690739
9,2020,0.527473,0.535998,0.692445


#13 Combined Model Season-by-Season Backtest

In [18]:
run_season_backtest(matchup_clean, baseline_features, "Combined Backtest")


Combined Backtest
Season 2011: Acc=0.647, AUC=0.640, LogLoss=0.642
Season 2012: Acc=0.641, AUC=0.660, LogLoss=0.632
Season 2013: Acc=0.622, AUC=0.659, LogLoss=0.642
Season 2014: Acc=0.656, AUC=0.683, LogLoss=0.629
Season 2015: Acc=0.647, AUC=0.679, LogLoss=0.627
Season 2016: Acc=0.617, AUC=0.640, LogLoss=0.652
Season 2017: Acc=0.626, AUC=0.668, LogLoss=0.635
Season 2018: Acc=0.629, AUC=0.648, LogLoss=0.644
Season 2019: Acc=0.608, AUC=0.652, LogLoss=0.654
Season 2020: Acc=0.592, AUC=0.631, LogLoss=0.662
Season 2021: Acc=0.609, AUC=0.633, LogLoss=0.661
Season 2022: Acc=0.595, AUC=0.593, LogLoss=0.673
Season 2023: Acc=0.597, AUC=0.671, LogLoss=0.655
Season 2024: Acc=0.633, AUC=0.672, LogLoss=0.640
Season 2025: Acc=0.641, AUC=0.692, LogLoss=0.628


,season,acc,auc,logloss
0,2011,0.647114,0.640240,0.642027
1,2012,0.640791,0.659551,0.631638
2,2013,0.621683,0.659462,0.642114
3,2014,0.655988,0.682725,0.629293
4,2015,0.646657,0.678852,0.627266
5,2016,0.617265,0.639547,0.651577
6,2017,0.625762,0.667507,0.635345
7,2018,0.628811,0.648075,0.643546
8,2019,0.608027,0.652110,0.653654
9,2020,0.591802,0.631219,0.661834


In [19]:
run_season_backtest(matchup_clean, full_features, "Combined Backtest")


Combined Backtest
Season 2011: Acc=0.651, AUC=0.653, LogLoss=0.638
Season 2012: Acc=0.644, AUC=0.671, LogLoss=0.625
Season 2013: Acc=0.626, AUC=0.668, LogLoss=0.638
Season 2014: Acc=0.658, AUC=0.689, LogLoss=0.628
Season 2015: Acc=0.652, AUC=0.697, LogLoss=0.616
Season 2016: Acc=0.633, AUC=0.653, LogLoss=0.645
Season 2017: Acc=0.649, AUC=0.671, LogLoss=0.633
Season 2018: Acc=0.649, AUC=0.672, LogLoss=0.632
Season 2019: Acc=0.616, AUC=0.660, LogLoss=0.650
Season 2020: Acc=0.608, AUC=0.646, LogLoss=0.655
Season 2021: Acc=0.610, AUC=0.641, LogLoss=0.658
Season 2022: Acc=0.600, AUC=0.616, LogLoss=0.664
Season 2023: Acc=0.633, AUC=0.682, LogLoss=0.642
Season 2024: Acc=0.657, AUC=0.703, LogLoss=0.624
Season 2025: Acc=0.656, AUC=0.708, LogLoss=0.617


,season,acc,auc,logloss
0,2011,0.650838,0.653392,0.637998
1,2012,0.643836,0.670517,0.624849
2,2013,0.626232,0.668485,0.638395
3,2014,0.657513,0.689275,0.627687
4,2015,0.651976,0.696524,0.615548
5,2016,0.632544,0.652929,0.645061
6,2017,0.648628,0.671373,0.632959
7,2018,0.648628,0.671888,0.632342
8,2019,0.615713,0.660392,0.649566
9,2020,0.608027,0.645869,0.654864


#14 Predicting the playoffs using regular season data

In [20]:
def train_on_regular_test_on_playoffs(reg_data, po_data, features):

    print("\n" + "="*60)
    print("Train: Regular Season to Test: Playoffs")
    print("="*60)

    # ----------------------------
    # Train on regular season
    # ----------------------------
    X_train = reg_data[features]
    y_train = reg_data["win"]

    # ----------------------------
    # Test on playoffs
    # ----------------------------
    X_test = po_data[features]
    y_test = po_data["win"]

    # ----------------------------
    # Scaling (fit only on train)
    # ----------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ----------------------------
    # Model
    # ----------------------------
    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ----------------------------
    # Predictions
    # ----------------------------
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    # ----------------------------
    # Evaluation
    # ----------------------------
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)
    ll = log_loss(y_test, prob)

    print("Accuracy :", round(acc, 4))
    print("ROC AUC  :", round(auc, 4))
    print("Log Loss :", round(ll, 4))

    return model

In [21]:
train_on_regular_test_on_playoffs(
    matchup_reg,
    matchup_po,
    full_features
)


Train: Regular Season to Test: Playoffs
Accuracy : 0.612
ROC AUC  : 0.6122
Log Loss : 0.657


LogisticRegression(C=0.1, penalty='l1', random_state=42, solver='liblinear')

#15 Predicting the playoffs using data from the same season's regular season

In [22]:
def train_reg_test_playoff_same_season(data, features):

    print("\n" + "="*60)
    print("Regular Season to Playoffs (Same Season)")
    print("="*60)

    results = []

    seasons = sorted(data["season"].unique())

    for season in seasons:

        reg = data[(data["season"] == season) & (data["is_playoff"] == 0)]
        po  = data[(data["season"] == season) & (data["is_playoff"] == 1)]

        # skip seasons without playoffs
        if len(po) == 0 or len(reg) == 0:
            continue

        X_train = reg[features]
        y_train = reg["win"]

        X_test = po[features]
        y_test = po["win"]

        # scaling
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # model
        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.1,
            random_state=42
        )

        model.fit(X_train, y_train)

        # predictions
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)
        ll = log_loss(y_test, prob)

        print(f"\nSeason {season}")
        
        print("Regular games:", len(reg))
        print("Playoff games:", len(po))
        print("Accuracy :", round(acc, 4))
        print("ROC AUC  :", round(auc, 4))
        print("Log Loss :", round(ll, 4))

        results.append([season, acc, auc, ll])

    return pd.DataFrame(results, columns=["season", "accuracy", "roc_auc", "log_loss"])

In [23]:
playoff_transfer_results = train_reg_test_playoff_same_season(
    matchup_clean,
    full_features
)

playoff_transfer_results


Regular Season to Playoffs (Same Season)

Season 2010
Regular games: 1213
Playoff games: 81
Accuracy : 0.6667
ROC AUC  : 0.6413
Log Loss : 0.6265

Season 2011
Regular games: 990
Playoff games: 84
Accuracy : 0.6667
ROC AUC  : 0.6537
Log Loss : 0.6179

Season 2012
Regular games: 1229
Playoff games: 85
Accuracy : 0.6471
ROC AUC  : 0.5651
Log Loss : 0.6618

Season 2013
Regular games: 1230
Playoff games: 89
Accuracy : 0.5281
ROC AUC  : 0.4974
Log Loss : 0.711

Season 2014
Regular games: 1230
Playoff games: 81
Accuracy : 0.5556
ROC AUC  : 0.6105
Log Loss : 0.6674

Season 2015
Regular games: 1230
Playoff games: 86
Accuracy : 0.6163
ROC AUC  : 0.6336
Log Loss : 0.6362

Season 2016
Regular games: 1230
Playoff games: 79
Accuracy : 0.5949
ROC AUC  : 0.6327
Log Loss : 0.6495

Season 2017
Regular games: 1230
Playoff games: 82
Accuracy : 0.6585
ROC AUC  : 0.5927
Log Loss : 0.6499

Season 2018
Regular games: 1230
Playoff games: 82
Accuracy : 0.6463
ROC AUC  : 0.663
Log Loss : 0.6518

Season 2019
Reg

,season,accuracy,roc_auc,log_loss
0,2010,0.666667,0.641289,0.626481
1,2011,0.666667,0.653671,0.617866
2,2012,0.647059,0.565114,0.661753
3,2013,0.528090,0.497436,0.710970
4,2014,0.555556,0.610480,0.667392
5,2015,0.616279,0.633621,0.636181
6,2016,0.594937,0.632680,0.649480
7,2017,0.658537,0.592672,0.649860
8,2018,0.646341,0.663043,0.651826
9,2019,0.582278,0.578846,0.695987
